# 01 — Recorded defense demo: Flip/ReFlip on Llama-3-8B

**Attach before running:** the two datasets built by notebook `00`
(`llama3-rtn-demo`, `llama3-rtn-reflip-demo`). Have your HF token ready — you
will be prompted to paste it (or set the `HF_TOKEN` Kaggle Secret as fallback).
Accelerator: **GPU T4 x2** - Internet **On**.

Timeline (record the screen, cut the eval waits in editing):
1. **Demo 1** - standalone Q-K case study (layer 8, GQA group 3): error table + Kneedle figures.
2. **Demo 2** - full precision vs RTN vs RTN+ReFlip: WikiText perplexity + ARC-Easy subset,
   then the concrete questions ReFlip fixed.


In [ ]:
# --- Setup: repo, deps, checkpoint paths ---
import glob, os, subprocess, sys

if not os.path.exists("/kaggle/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Itvc0110/Reflip-Flip-on-QKV-.git", "/kaggle/tmp/repo"], check=True)
os.chdir("/kaggle/tmp/repo")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.45,<4.53", "datasets>=2.20,<3.0", "accelerate",
                "sentencepiece", "kneed", "matplotlib", "pandas"], check=True)

def find_input(pattern):
    hits = glob.glob(f"/kaggle/input/*/{pattern}") + glob.glob(f"/kaggle/input/{pattern}")
    assert hits, f"attach the dataset containing '{pattern}'"
    return hits[0]

RTN_PATH    = find_input("llama3_rtn")
REFLIP_PATH = find_input("llama3_rtn_reflip")
DEMO1_DIR   = find_input("demo1")
print("RTN checkpoint:   ", RTN_PATH)
print("ReFlip checkpoint:", REFLIP_PATH)
print("Demo-1 artifacts: ", DEMO1_DIR)

## Demo 1 — Standalone Q–K case study (Llama-3-8B, layer 8, GQA group 3)

One GQA group under the microscope: 4 query heads sharing one key head, asymmetric INT4
(group size 128). Three successive states — **RTN → Flip → Flip + ReFlip** — evaluated on the
exact scalar Q–K surrogate each stage optimizes. Artifacts were produced by
`xspot.py` → `fast_quantize_qkv.py` (same pipeline as the thesis).

In [ ]:
# --- Demo 1: per-head error table (matches thesis Tables 4.2/4.3) ---
import subprocess, sys
subprocess.run([sys.executable, "tools/summarize_qkv_results.py",
                "--npz", f"{DEMO1_DIR}/quantization_results.npz"], check=True)

In [ ]:
# --- Demo 1: figures (regenerated live from the run's npz) ---
import subprocess, sys
from IPython.display import Image, display

subprocess.run([sys.executable, "tools/plot_kneedle_sensitivity.py",
                "--npz", f"{DEMO1_DIR}/quantization_results.npz",
                "--out", "/kaggle/working/reflip_kneedle.png"], check=True)
subprocess.run([sys.executable, "tools/plot_flip_activation_kneedle.py",
                "--npz", f"{DEMO1_DIR}/quantization_results.npz",
                "--out", "/kaggle/working/flip_kneedle.png"], check=True)

display(Image("/kaggle/working/flip_kneedle.png", width=760))
display(Image("/kaggle/working/reflip_kneedle.png", width=760))
display(Image(f"{DEMO1_DIR}/attention_quantization_analysis.png", width=860))

## Demo 2 — Quick 3-way comparison: full precision vs RTN vs RTN + ReFlip

Same seeded pool of **ARC-Easy test questions**, scored by log-likelihood over the answer
choices (the benchmark's own decision rule). One model in memory at a time. The script
auto-selects the most explainable questions — **RTN answers wrong, RTN + ReFlip answers
right** — and logs per-model answers vs the gold answer, per-question inference time,
generation speed, and memory footprint.

In [ ]:
# --- Download the full-precision reference (scratch dir, ~16 GB) ---
from getpass import getpass
from huggingface_hub import login, snapshot_download

# Primary: type/paste the token at the prompt (input hidden).
# Fallback (non-interactive runs): Kaggle Secret named HF_TOKEN.
try:
    token = getpass("Paste your Hugging Face access token (input hidden): ").strip()
except Exception:
    token = ""
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print("(using HF_TOKEN Kaggle Secret)")
login(token=token)

FP_PATH = "/kaggle/tmp/models/Llama-3-8B"
snapshot_download("meta-llama/Meta-Llama-3-8B", local_dir=FP_PATH,
                  ignore_patterns=["original/*", "*.pth"])
print("full-precision model at", FP_PATH)

In [ ]:
# --- Quick compare (the long-ish cell: ~5-6 min per model incl. load) ---
import subprocess, sys
subprocess.run([sys.executable, "Demo/demo_quick_compare.py",
                "--fp-path", FP_PATH,
                "--rtn-path", RTN_PATH,
                "--reflip-path", REFLIP_PATH,
                "--n-questions", "100",
                "--showcase", "5",
                "--gen-tokens", "64",
                "--out-json", "/kaggle/working/quick_compare.json"], check=True)

In [ ]:
# --- Summary table (clean view for the recording) ---
import json
import pandas as pd

data = json.load(open("/kaggle/working/quick_compare.json"))
rows = []
for label, r in data["results"].items():
    rows.append({
        "model": label,
        "accuracy": round(r["accuracy"], 4),
        "mean s/question": round(r["mean_time_s"], 3),
        "gen tokens/s": round(r["gen_tokens_per_s"], 1),
        "peak VRAM (GB)": round(r["peak_vram_gb"], 1),
        "checkpoint (GB)": round(r["checkpoint_disk_gb"], 1),
    })
display(pd.DataFrame(rows).set_index("model"))
print(f"questions RTN got wrong that ReFlip fixed: {data['n_fixed_by_reflip']}")
print("Thesis Table 4.7 reference (full benchmark): RTN avg 0.6924 -> RTN+ReFlip 0.7033 (+1.09 pts)")

## Wrap-up

- **Demo 1**: the standalone case study reproduces the thesis numbers live — scalar Q–K error
  0.1428 → 0.0701 (Flip) → 0.0479 (Flip + ReFlip) while touching ~0.8% of weights + 64 moves.
- **Demo 2**: on the same ARC-Easy test questions, RTN + ReFlip answers more questions correctly
  than RTN — including the showcased questions where the flip from wrong to right is explicit —
  at **identical inference time and memory** (the refinement adds zero runtime overhead).

*(Fresh run on a seeded question pool with reduced calibration; presented as directionally
consistent with the thesis tables, not a bit-exact reproduction.)*